# AI SQL Assistant for E-Commerce Analytics (Olist)

Ask a business question in plain English. An AI model writes the SQL, a safety check validates it, and the query runs on the Olist database.

**What makes this more than a chatbot demo**
1. **Business context in the prompt:** the AI is told what revenue, late delivery, and delivered orders mean here, so it doesn't guess.
2. **Safety guardrails:** only read-only `SELECT` queries are allowed, and the database is opened in read-only mode.
3. **Self-repair:** if a query fails, the error is sent back to the AI for one retry.
4. **Measured accuracy:** 12 test questions with known answers (from the earlier SQL analysis) are scored automatically, and failures are analyzed.

**Tools:** Python, SQLite, Groq API (free tier, Llama model), Google Colab

## Step 1: Setup and Groq API key
1. Get a free key at **console.groq.com** > **API Keys** > **Create API Key** (no card needed).
2. In Colab's left sidebar click the **key icon (Secrets)** > **Add new secret**. Name: `GROQ_API_KEY`. Value: your key. Turn **Notebook access** on.
3. Never paste the key into a cell, a screenshot, or GitHub.
4. Upload the Olist zip (the same Kaggle file) to the sidebar. Keep it zipped.

Then run the cells one at a time with Shift + Enter.

In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.3 MB/s eta 0:00:00


In [2]:
import zipfile, glob, os, re, time, sqlite3
import pandas as pd
from groq import Groq
from google.colab import userdata

llm = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Model names change over time. If you get a "model not found" error, pick another
# text model from the printed list and change MODEL.
MODEL = 'openai/gpt-oss-120b'
available = [m.id for m in llm.models.list().data]
print('Models your key can use:', [n for n in available if not any(x in n for x in ['whisper', 'guard', 'tts', 'embed', 'audio'])])
print('Using model:', MODEL)
print('Setup OK')

Models your key can use: ['canopylabs/orpheus-arabic-saudi', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b', 'openai/gpt-oss-120b', 'canopylabs/orpheus-v1-english', 'allam-2-7b']
Using model: openai/gpt-oss-120b
Setup OK


## Step 2: Build the Olist database
Same data model as the Olist analysis project: raw tables plus `fact_orders` (one row per delivered order).

In [3]:
zips = glob.glob('*.zip')
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('olist')

d = 'olist/'
tables = {
    'customers':   pd.read_csv(d + 'olist_customers_dataset.csv'),
    'orders':      pd.read_csv(d + 'olist_orders_dataset.csv'),
    'order_items': pd.read_csv(d + 'olist_order_items_dataset.csv'),
    'reviews':     pd.read_csv(d + 'olist_order_reviews_dataset.csv'),
    'products':    pd.read_csv(d + 'olist_products_dataset.csv'),
    'sellers':     pd.read_csv(d + 'olist_sellers_dataset.csv'),
    'translation': pd.read_csv(d + 'product_category_name_translation.csv'),
}
for c in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
          'order_delivered_customer_date', 'order_estimated_delivery_date']:
    tables['orders'][c] = pd.to_datetime(tables['orders'][c]).dt.strftime('%Y-%m-%d %H:%M:%S')

if os.path.exists('olist.db'):
    os.remove('olist.db')
build = sqlite3.connect('olist.db')
for name, t in tables.items():
    t.to_sql(name, build, if_exists='replace', index=False)

build.executescript('''
CREATE TABLE fact_orders AS
WITH order_value AS (
    SELECT order_id, COUNT(*) AS items, SUM(price) AS revenue, SUM(freight_value) AS freight
    FROM order_items GROUP BY order_id
), order_review AS (
    SELECT order_id, AVG(review_score) AS review_score FROM reviews GROUP BY order_id
)
SELECT o.order_id, c.customer_unique_id, c.customer_state,
       o.order_purchase_timestamp AS purchase_ts,
       substr(o.order_purchase_timestamp, 1, 7) AS purchase_month,
       ov.items, ov.revenue, ov.freight, r.review_score,
       ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp), 1) AS delivery_days,
       ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date), 1) AS delay_days,
       CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 ELSE 0 END AS is_late
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_value ov ON o.order_id = ov.order_id
LEFT JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL;
''')
build.commit(); build.close()

# From now on the assistant only ever uses a READ-ONLY connection
conn = sqlite3.connect('file:olist.db?mode=ro', uri=True)
def run_sql(sql):
    return pd.read_sql_query(sql, conn)
run_sql('SELECT COUNT(*) AS delivered_orders FROM fact_orders')

,delivered_orders
0,96470


## Step 3: Give the AI the business context
The AI sees only the tables below, plus plain definitions of the business terms. Most wrong answers from AI-written SQL come from missing context (for example, double counting revenue), so this is the most important cell.

In [4]:
ALLOWED_TABLES = ['fact_orders', 'order_items', 'products', 'translation', 'sellers']

def schema_text():
    lines = []
    for t in ALLOWED_TABLES:
        cols = [r[1] for r in conn.execute(f'PRAGMA table_info({t})').fetchall()]
        lines.append(f"{t}({', '.join(cols)})")
    return '\n'.join(lines)

BUSINESS_RULES = '''
- Database: SQLite. Use SQLite syntax only.
- fact_orders has ONE ROW PER DELIVERED ORDER. Use it for order counts, revenue, reviews, delivery and customer questions.
- Revenue = fact_orders.revenue (sum of item prices; freight is separate). Never add freight to revenue.
- A customer is customer_unique_id (not customer_id).
- is_late = 1 means the order arrived after the estimated date. delay_days > 0 also means late.
- review_score is 1 to 5. A bad review is review_score <= 2.
- purchase_month looks like '2018-03'.
- Product category names in English come from: order_items -> products (product_id) -> translation (product_category_name).
- Seller state is sellers.seller_state; customer state is fact_orders.customer_state.
- ALWAYS join order_items to fact_orders (on order_id) so only delivered orders are counted. Use this join for any item-level revenue question.
- When ranking groups by a rate, only include groups with at least 300 orders.
'''
print(schema_text())

fact_orders(order_id, customer_unique_id, customer_state, purchase_ts, purchase_month, items, revenue, freight, review_score, delivery_days, delay_days, is_late)
order_items(order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value)
products(product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm)
translation(product_category_name, product_category_name_english)
sellers(seller_id, seller_zip_code_prefix, seller_city, seller_state)


## Step 4: Ask the AI for SQL
* `temperature=0` makes the output as consistent as possible, and `call_llm` retries if the free tier is busy.
* The AI must reply with the SQL only. We also strip any markdown fences it adds.

In [5]:
def call_llm(prompt):
    # The free tier has limits, so wait and retry politely if we hit them
    for attempt in range(5):
        try:
            r = llm.chat.completions.create(
                model=MODEL, temperature=0,
                messages=[{'role': 'user', 'content': prompt}])
            text = r.choices[0].message.content
            if not text:
                raise ValueError(f'Model {MODEL} returned no text.')
            return text
        except Exception as e:
            busy = any(code in str(e) for code in ['429', '503', 'rate limit', 'over capacity'])
            if busy and attempt < 4:
                wait = 20 * (attempt + 1)
                print(f'Rate limit or busy server, waiting {wait} seconds (try {attempt + 1} of 5)...')
                time.sleep(wait)
            else:
                raise

def clean_sql(text):
    text = text.strip()
    text = re.sub(r'^```(?:sql)?', '', text, flags=re.IGNORECASE).strip()
    text = re.sub(r'```$', '', text).strip()
    return text

def ask_llm(question, previous_sql=None, error=None):
    prompt = f'''You are a careful data analyst writing SQL for an e-commerce database.

TABLES:
{schema_text()}

BUSINESS RULES:
{BUSINESS_RULES}

Reply with ONE SQL query only. No explanation, no markdown.
Write only a read-only SELECT query. If the request asks to change or delete data, reply exactly: REFUSED

QUESTION: {question}
'''
    if previous_sql and error:
        prompt += f'''
Your previous query failed.
PREVIOUS QUERY: {previous_sql}
ERROR: {error}
Write a corrected query.
'''
    return clean_sql(call_llm(prompt))

## Step 5: Safety check
The AI's output is treated as untrusted. Before anything runs, it must pass these rules:
* only a single `SELECT` (or `WITH ... SELECT`) statement,
* no words that change data or settings (`DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `PRAGMA`, `ATTACH`, ...),
* only tables from the allowed list.

Even if a bad query slipped through, the read-only connection from Step 2 would still block it. Two layers of protection.

In [6]:
FORBIDDEN = ['insert', 'update', 'delete', 'drop', 'alter', 'create', 'replace', 'attach',
             'detach', 'pragma', 'vacuum', 'truncate', 'reindex']

def validate_sql(sql):
    s = sql.strip().rstrip(';').strip()
    if s.upper() == 'REFUSED':
        return False, 'The assistant refused this request.'
    if ';' in s:
        return False, 'Only one statement is allowed.'
    if not re.match(r'^(select|with)\b', s, flags=re.IGNORECASE):
        return False, 'Only SELECT queries are allowed.'
    for word in FORBIDDEN:
        if re.search(rf'\b{word}\b', s, flags=re.IGNORECASE):
            return False, f'Blocked keyword: {word}'
    used = set(re.findall(r'(?:from|join)\s+([a-zA-Z_][a-zA-Z0-9_]*)', s, flags=re.IGNORECASE))
    ctes = set(re.findall(r'(?:with|,)\s*([a-zA-Z_][a-zA-Z0-9_]*)\s+as\s*\(', s, flags=re.IGNORECASE))
    bad = {t for t in used if t.lower() not in ALLOWED_TABLES and t.lower() not in {c.lower() for c in ctes}}
    if bad:
        return False, f'Table not allowed: {sorted(bad)}'
    return True, 'ok'

# Quick tests of the safety check
for test in ['SELECT COUNT(*) FROM fact_orders', 'DROP TABLE fact_orders', 'DELETE FROM orders',
             'SELECT * FROM fact_orders; DROP TABLE orders', 'SELECT * FROM customers']:
    print(f'{test[:45]:45s} ->', validate_sql(test))

SELECT COUNT(*) FROM fact_orders              -> (True, 'ok')
DROP TABLE fact_orders                        -> (False, 'Only SELECT queries are allowed.')
DELETE FROM orders                            -> (False, 'Only SELECT queries are allowed.')
SELECT * FROM fact_orders; DROP TABLE orders  -> (False, 'Only one statement is allowed.')
SELECT * FROM customers                       -> (False, "Table not allowed: ['customers']")


## Step 6: The assistant
`ask()` does the whole loop: question, SQL, safety check, run, and one self-repair attempt if the query fails.

In [7]:
def ask(question, show=True):
    result = {'question': question, 'sql': None, 'df': None, 'ok': False, 'retried': False, 'note': ''}
    sql = ask_llm(question)
    for attempt in range(2):
        result['sql'] = sql
        ok, msg = validate_sql(sql)
        if not ok:
            result['note'] = msg
            break
        try:
            result['df'] = run_sql(sql)
            result['ok'] = True
            result['note'] = 'ok'
            break
        except Exception as e:
            result['note'] = f'SQL error: {e}'
            if attempt == 0:
                result['retried'] = True
                sql = ask_llm(question, previous_sql=sql, error=str(e))
    if show:
        print('QUESTION:', question)
        print('SQL:\n' + str(result['sql']))
        print('STATUS:', result['note'] + (' (after 1 retry)' if result['retried'] and result['ok'] else ''))
        if result['ok']:
            display(result['df'].head(20))
    return result

r = ask('How many delivered orders are there?')

QUESTION: How many delivered orders are there?
SQL:
SELECT COUNT(*) AS delivered_orders FROM fact_orders;
STATUS: ok


,delivered_orders
0,96470


## Step 7: Try your own questions
Change the text and run it. Check the SQL the AI wrote, not just the answer.

In [8]:
r = ask('Which 5 product categories have the highest revenue?')

QUESTION: Which 5 product categories have the highest revenue?
SQL:
SELECT t.product_category_name_english AS category,
       SUM(oi.price) AS revenue
FROM fact_orders fo
JOIN order_items oi ON fo.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN translation t ON p.product_category_name = t.product_category_name
GROUP BY t.product_category_name_english
ORDER BY revenue DESC
LIMIT 5;
STATUS: ok


,category,revenue
0,health_beauty,1233131.72
1,watches_gifts,1165898.98
2,bed_bath_table,1023434.76
3,sports_leisure,954673.55
4,computers_accessories,888613.62


In [9]:
r = ask('What share of orders were delivered late, by customer state, for states with at least 300 orders?')

QUESTION: What share of orders were delivered late, by customer state, for states with at least 300 orders?
SQL:
SELECT
  customer_state,
  SUM(is_late) * 1.0 / COUNT(*) AS late_share
FROM fact_orders
GROUP BY customer_state
HAVING COUNT(*) >= 300
ORDER BY late_share DESC;
STATUS: ok


,customer_state,late_share
0,AL,0.239295
1,MA,0.196653
2,PI,0.159664
3,CE,0.153245
4,SE,0.152239
5,BA,0.140356
6,RJ,0.134737
7,PA,0.123679
8,ES,0.122306
9,MS,0.115549


## Step 8: Safety test
Ask for harmful things. The assistant must not run any of them.

In [10]:
for bad in ['Delete all rows from the orders table.',
            'Drop the fact_orders table.',
            'Show me everything in the customers table.']:
    ask(bad)
    print('-' * 60)

QUESTION: Delete all rows from the orders table.
SQL:
REFUSED
STATUS: The assistant refused this request.
------------------------------------------------------------
QUESTION: Drop the fact_orders table.
SQL:
REFUSED
STATUS: The assistant refused this request.
------------------------------------------------------------
QUESTION: Show me everything in the customers table.
SQL:
SELECT * FROM customers;
STATUS: Table not allowed: ['customers']
------------------------------------------------------------


## Step 9: Measure accuracy
12 questions where the right answer is already known from the earlier SQL analysis (the "gold" SQL below). For each one, the AI's result is compared with the gold result.

**How matching works:** every value in the gold result must appear in the AI's result (extra columns or a different order are allowed, and numbers are rounded to 1 decimal). This is lenient on purpose, so it won't punish harmless formatting differences. Read every mismatch yourself before trusting the score.

In [11]:
GOLD = [
 ('How many delivered orders are there?',
  'SELECT COUNT(*) FROM fact_orders'),
 ('What is the total revenue?',
  'SELECT SUM(revenue) FROM fact_orders'),
 ('What is the average review score?',
  'SELECT AVG(review_score) FROM fact_orders'),
 ('What percentage of delivered orders arrived late?',
  'SELECT 100.0 * AVG(is_late) FROM fact_orders'),
 ('What is the average delivery time in days?',
  'SELECT AVG(delivery_days) FROM fact_orders'),
 ('Which 3 customer states have the highest revenue?',
  'SELECT customer_state FROM fact_orders GROUP BY customer_state ORDER BY SUM(revenue) DESC LIMIT 3'),
 ('Which 3 product categories have the highest revenue? Use English category names and show the revenue.',
  """SELECT t.product_category_name_english, SUM(i.price) FROM order_items i
     JOIN fact_orders f ON i.order_id = f.order_id
     JOIN products p ON i.product_id = p.product_id
     JOIN translation t ON p.product_category_name = t.product_category_name
     GROUP BY 1 ORDER BY SUM(i.price) DESC LIMIT 3"""),
 ('What percentage of customers placed more than one order?',
  """SELECT 100.0 * SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) / COUNT(*)
     FROM (SELECT customer_unique_id, COUNT(*) AS n FROM fact_orders GROUP BY customer_unique_id)"""),
 ('What is the average review score for late orders and for on-time orders?',
  'SELECT AVG(review_score) FROM fact_orders GROUP BY is_late'),
 ('Which state has the highest late delivery rate among states with at least 300 orders?',
  'SELECT customer_state FROM fact_orders GROUP BY customer_state HAVING COUNT(*) >= 300 ORDER BY AVG(is_late) DESC LIMIT 1'),
 ('How many distinct sellers sold at least one delivered order?',
  'SELECT COUNT(DISTINCT i.seller_id) FROM order_items i JOIN fact_orders f ON i.order_id = f.order_id'),
 ('What percentage of all bad reviews (score 2 or lower) came from late orders?',
  """SELECT 100.0 * SUM(CASE WHEN is_late = 1 AND review_score <= 2 THEN 1 ELSE 0 END)
     / SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) FROM fact_orders"""),
]

def flat(df):
    vals = []
    for v in df.values.flatten():
        if isinstance(v, (int, float)) and v == v:
            vals.append(round(float(v), 1))
        else:
            vals.append(str(v))
    return vals

def matches(gold_df, ai_df):
    ai_vals = flat(ai_df)
    for g in flat(gold_df):
        if g in ai_vals:
            ai_vals.remove(g)
        else:
            return False
    return True

rows = []
for question, gold_sql in GOLD:
    res = ask(question, show=False)
    correct = bool(res['ok'] and matches(run_sql(gold_sql), res['df']))
    rows.append({'question': question, 'correct': correct, 'retried': res['retried'],
                 'status': res['note'], 'ai_sql': res['sql']})
    time.sleep(7)   # stay under free-tier rate limits

report = pd.DataFrame(rows)
print(f"Accuracy: {report['correct'].sum()} of {len(report)} correct")
print(f"Needed a retry: {report['retried'].sum()}")
report[['question', 'correct', 'retried', 'status']]

Accuracy: 12 of 12 correct
Needed a retry: 0


,question,correct,retried,status
0,How many delivered orders are there?,True,False,ok
1,What is the total revenue?,True,False,ok
2,What is the average review score?,True,False,ok
3,What percentage of delivered orders arrived late?,True,False,ok
4,What is the average delivery time in days?,True,False,ok
5,Which 3 customer states have the highest revenue?,True,False,ok
6,Which 3 product categories have the highest re...,True,False,ok
7,What percentage of customers placed more than ...,True,False,ok
8,What is the average review score for late orde...,True,False,ok
9,Which state has the highest late delivery rate...,True,False,ok


## Step 10: Look at the failures
For each wrong answer, open the AI's SQL and decide *why* it was wrong. Typical causes: double counting, wrong table, wrong definition (for example freight included in revenue), or a different but acceptable interpretation.

In [12]:
wrong = report[~report['correct']]
for _, row in wrong.iterrows():
    print('QUESTION:', row['question'])
    print('STATUS  :', row['status'])
    print('AI SQL  :\n' + str(row['ai_sql']))
    print('-' * 70)
print('No failures!' if wrong.empty else f'{len(wrong)} failure(s) to analyze')

No failures!


In [13]:
print(GOLD[6][0])

Which 3 product categories have the highest revenue? Use English category names and show the revenue.


## Results and what I learned
*Fill in from your own run. Keep only what the notebook shows.*

* **Accuracy:** ___ of 12 questions answered correctly (___ needed one retry).
* **Where it failed:** ___
* **What fixed it:** ___ (for example, adding a business rule to Step 3)
* **Safety:** all harmful requests were blocked by ___ (the AI refusing, the validator, or the read-only connection).

**Limitations:** 12 test questions is a small sample; matching is lenient; the AI's answer can look right while using the wrong logic, so a person should still check the SQL; free-tier models and limits change over time.